In [2]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

file_path="medical_conditions_dataset.csv"
df=pd.read_csv(file_path)

In [3]:
print(df.head)

<bound method NDFrame.head of          id  full_name   age  gender smoking_status        bmi  \
0         1   User0001   NaN    male     Non-Smoker        NaN   
1         2   User0002  30.0    male     Non-Smoker        NaN   
2         3   User0003  18.0    male     Non-Smoker  35.612486   
3         4   User0004   NaN    male     Non-Smoker        NaN   
4         5   User0005  76.0    male     Non-Smoker        NaN   
...     ...        ...   ...     ...            ...        ...   
9995   9996   User9996   NaN    male     Non-Smoker  25.029002   
9996   9997   User9997   NaN    male     Non-Smoker  27.017487   
9997   9998   User9998  23.0    male         Smoker        NaN   
9998   9999   User9999   NaN  female     Non-Smoker        NaN   
9999  10000  User10000  27.0    male     Non-Smoker  25.454891   

      blood_pressure  glucose_levels  condition  
0                NaN             NaN  Pneumonia  
1         105.315064             NaN   Diabetic  
2                NaN       

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              10000 non-null  int64  
 1   full_name       10000 non-null  object 
 2   age             5445 non-null   float64
 3   gender          10000 non-null  object 
 4   smoking_status  10000 non-null  object 
 5   bmi             4652 non-null   float64
 6   blood_pressure  3766 non-null   float64
 7   glucose_levels  4756 non-null   float64
 8   condition       10000 non-null  object 
dtypes: float64(4), int64(1), object(4)
memory usage: 703.3+ KB


In [5]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
df_cleaned = df.drop(columns=['id', 'full_name'])

numerical_cols = ['age', 'bmi', 'blood_pressure', 'glucose_levels']
df_cleaned['gender'] = df_cleaned['gender'].str.title()
df_cleaned['smoking_status'] = df_cleaned['smoking_status'].str.replace('-', ' ').str.title()
X_num = df_cleaned[numerical_cols]
X_cat = df_cleaned.drop(columns=numerical_cols)

#knn imputation
imputer = KNNImputer(n_neighbors=5) 
X_imputed_array = imputer.fit_transform(X_num)
X_imputed = pd.DataFrame(X_imputed_array, columns=numerical_cols)
X_final_cleaned = pd.concat([X_imputed, X_cat], axis=1)

print("--- Data Info After kNN Imputation ---")
print(X_final_cleaned.info())

print("\n--- Cleaned Data Head (showing imputed values) ---")
print(X_final_cleaned.head())

output_filename = 'medical_conditions_dataset_KNN_Imputed.csv'
X_final_cleaned.to_csv(output_filename, index=False)
print(f"\n the cleaned dataset has been successfully exported to: '{output_filename}'")
print("\n--- Exported Data Head (for confirmation of imputed/cleaned state) ---")
print(X_final_cleaned.head())

--- Data Info After kNN Imputation ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             10000 non-null  float64
 1   bmi             10000 non-null  float64
 2   blood_pressure  10000 non-null  float64
 3   glucose_levels  10000 non-null  float64
 4   gender          10000 non-null  object 
 5   smoking_status  10000 non-null  object 
 6   condition       10000 non-null  object 
dtypes: float64(4), object(3)
memory usage: 547.0+ KB
None

--- Cleaned Data Head (showing imputed values) ---
         age        bmi  blood_pressure  glucose_levels gender smoking_status  \
0  53.541598  27.423420      135.209429      135.219608   Male     Non Smoker   
1  30.000000  28.924770      105.315064      148.837937   Male     Non Smoker   
2  18.000000  35.612486      138.153310      153.485514   Male     Non Smoker   
3  54.000000  

In [6]:
output_filename = 'medical_conditions_dataset_KNN_Imputed.csv'
X_final_cleaned.to_csv(output_filename, index=False)
print(f"\n the cleaned dataset has been successfully exported to: '{output_filename}'")
print("\n--- Exported Data Head (for confirmation of imputed/cleaned state) ---")
print(X_final_cleaned.head())


 the cleaned dataset has been successfully exported to: 'medical_conditions_dataset_KNN_Imputed.csv'

--- Exported Data Head (for confirmation of imputed/cleaned state) ---
         age        bmi  blood_pressure  glucose_levels gender smoking_status  \
0  53.541598  27.423420      135.209429      135.219608   Male     Non Smoker   
1  30.000000  28.924770      105.315064      148.837937   Male     Non Smoker   
2  18.000000  35.612486      138.153310      153.485514   Male     Non Smoker   
3  54.000000  25.621843       99.119829      110.798413   Male     Non Smoker   
4  76.000000  26.551568      134.310935      155.190920   Male     Non Smoker   

   condition  
0  Pneumonia  
1   Diabetic  
2  Pneumonia  
3  Pneumonia  
4   Diabetic  


In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

#feature and target seperation
X = X_final_cleaned.drop(columns=['condition'])
y = X_final_cleaned['condition']

#one hot encoding for categorical features(gender and smoking status)
X_encoded = pd.get_dummies(X, columns=['gender', 'smoking_status'], drop_first=True)
numerical_cols = ['age', 'bmi', 'blood_pressure', 'glucose_levels']
scaler = StandardScaler()
X_encoded[numerical_cols] = scaler.fit_transform(X_encoded[numerical_cols])
X_final = X_encoded

#training and testing split
X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.3, random_state=42, stratify=y)
print(f"data split complete.training on {X_train.shape[0]} samples.")

data split complete.training on 7000 samples.


In [8]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier # Import corrected to avoid conflict
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import pandas as pd

#re extracting features and targets
X = X_final_cleaned.drop(columns=['condition'])
y_raw = X_final_cleaned['condition']

#one hot encoding categorical features
X_encoded = pd.get_dummies(X, columns=['gender', 'smoking_status'], drop_first=True)

numerical_cols = ['age', 'bmi', 'blood_pressure', 'glucose_levels']
scaler = StandardScaler()
X_encoded[numerical_cols] = scaler.fit_transform(X_encoded[numerical_cols])
X_final = X_encoded

#label encoding to fix XGBoost
le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)

target_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"Target Variable Encoding Map: {target_mapping}")


X_train, X_test, y_train_enc, y_test_enc = train_test_split( X_final, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded)

#re running model approaches
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    'XGBoost': XGBClassifier(objective='multi:softmax', n_estimators=100, use_label_encoder=False, eval_metric='mlogloss', random_state=42),
    'SVC': SVC(random_state=42, class_weight='balanced'),
    'KNeighbors': KNeighborsClassifier(n_neighbors=10),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced', solver='liblinear')
}

results = {}

print("\n--- Restarting Training and Evaluation for 5 Models ---")
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train_enc)
    y_pred_enc = model.predict(X_test)
    #performance metrics
    accuracy = accuracy_score(y_test_enc, y_pred_enc)
    f1 = f1_score(y_test_enc, y_pred_enc, average='weighted')
    results[name] = {'Accuracy': accuracy, 'F1 Score (Weighted)': f1}
    print(f"Completed {name}. Accuracy: {accuracy:.4f}")

#comparison table
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values(by='F1 Score (Weighted)', ascending=False)

print("\n final comparative performance analysis ")
print(results_df)

Target Variable Encoding Map: {'Cancer': 0, 'Diabetic': 1, 'Pneumonia': 2}

--- Restarting Training and Evaluation for 5 Models ---
Training Random Forest...
Completed Random Forest. Accuracy: 0.5377
Training XGBoost...


C:\Users\tejas\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:29:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Completed XGBoost. Accuracy: 0.5543
Training SVC...
Completed SVC. Accuracy: 0.3277
Training KNeighbors...
Completed KNeighbors. Accuracy: 0.5533
Training Logistic Regression...
Completed Logistic Regression. Accuracy: 0.6013

 final comparative performance analysis 
                     Accuracy  F1 Score (Weighted)
Random Forest        0.537667             0.477588
XGBoost              0.554333             0.465614
KNeighbors           0.553333             0.457853
Logistic Regression  0.601333             0.451626
SVC                  0.327667             0.350776


In [9]:
#hyper parameter tuning
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint as sp_randint
from xgboost import XGBClassifier
from sklearn.metrics import f1_score

#reinstantiating XGBoost with correct settings and object type
xgb_model_tuned = XGBClassifier(
    objective='multi:softmax',
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)

#search space
param_dist = {
    'n_estimators': sp_randint(100, 500),
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': sp_randint(3, 10),
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'gamma': [0, 0.1, 0.5, 1],
}

#randomised search
#50 different combinations
random_search = RandomizedSearchCV(
    estimator=xgb_model_tuned,
    param_distributions=param_dist,
    n_iter=50,
    scoring='f1_weighted',  
    cv=5,                   
    verbose=1,
    random_state=42,
    n_jobs=-1               
)

print("\nStarting Hyperparameter Tuning for XGBoost...")
#fitting the search to the encoded training data 
random_search.fit(X_train, y_train_enc)

#best model find
best_xgb_model = random_search.best_estimator_
best_score = random_search.best_score_
tuned_y_pred = best_xgb_model.predict(X_test)

#final metrics for optimised model
final_accuracy = accuracy_score(y_test_enc, tuned_y_pred)
final_f1_score = f1_score(y_test_enc, tuned_y_pred, average='weighted')

print("Hyperparameter Tuning and Final Evaluation Complete")
print(f"\nBest parameters found: {random_search.best_params_}")
print(f"Optimized XGBoost Accuracy on Test Set: **{final_accuracy:.4f}**")
print(f"Optimized XGBoost F1 Score on Test Set: **{final_f1_score:.4f}**")


Starting Hyperparameter Tuning for XGBoost...
Fitting 5 folds for each of 50 candidates, totalling 250 fits


C:\Users\tejas\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:31:45] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Hyperparameter Tuning and Final Evaluation Complete

Best parameters found: {'colsample_bytree': 0.8, 'gamma': 0.5, 'learning_rate': 0.2, 'max_depth': 4, 'n_estimators': 211, 'subsample': 0.6}
Optimized XGBoost Accuracy on Test Set: **0.5730**
Optimized XGBoost F1 Score on Test Set: **0.4663**


In [10]:
#catboost
import pandas as pd
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

X = X_final_cleaned.drop(columns=['condition'])
y = X_final_cleaned['condition']
categorical_features = ['gender', 'smoking_status']

#training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    loss_function='MultiClass',
    eval_metric='Accuracy',
    random_state=42,
    verbose=0, #suppress intermediate output
    cat_features=categorical_features, 
    class_weights=[1.0, 1.0, 1.0] 
)

print("Starting CatBoost training for maximum performance...")
cat_model.fit(X_train, y_train, early_stopping_rounds=20)

cat_y_pred = cat_model.predict(X_test)

#convert CatBoost predictions to a flat array
if isinstance(cat_y_pred, np.ndarray) and cat_y_pred.ndim > 1:
    cat_y_pred = cat_y_pred.flatten()

#final metrics
cat_accuracy = accuracy_score(y_test, cat_y_pred)
cat_f1_score = f1_score(y_test, cat_y_pred, average='weighted')
cat_class_report = classification_report(y_test, cat_y_pred)

print("CatBoost model training completed")
print(f"\n--- CatBoost performance on test set")
print(f"Overall accuracy: **{cat_accuracy:.4f}**")
print(f"Weighted F1 score: **{cat_f1_score:.4f}**")
print("\nClassification report:\n", cat_class_report)

Starting CatBoost training for maximum performance...
CatBoost model training completed

--- CatBoost performance on test set
Overall accuracy: **0.5977**
Weighted F1 score: **0.4526**

Classification report:
               precision    recall  f1-score   support

      Cancer       0.20      0.00      0.00       438
    Diabetic       0.60      0.99      0.75      1804
   Pneumonia       0.17      0.00      0.01       758

    accuracy                           0.60      3000
   macro avg       0.32      0.33      0.25      3000
weighted avg       0.43      0.60      0.45      3000



In [11]:
#catboost 2
X = X_final_cleaned.drop(columns=['condition'])
y = X_final_cleaned['condition']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

#calculate the raw counts and then the inverse ratio of class frequency
class_counts = y_train.value_counts()
max_count = class_counts.max()
class_weights = {
    cls: max_count / count 
    for cls, count in class_counts.items()
}

# the class weights need to be mapped to the target classes in the training data
# CatBoost expects the weights as a list in the order of unique classes it encounters.
# the internal mapping CatBoost uses is alphabetical: Cancer:0, Diabetic:1, Pneumonia:2
catboost_weights = [
    class_weights['Cancer'], 
    class_weights['Diabetic'], 
    class_weights['Pneumonia']
]
print(f"Calculated CatBoost Class Weights (Order: Cancer, Diabetic, Pneumonia): {catboost_weights}")

#initialize CatBoostClassifier with explicit class_weights
cat_model_weighted = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,  # increase learning rate slightly for faster convergenc
    loss_function='MultiClass',
    eval_metric='Accuracy',
    random_state=42,
    verbose=0,
    cat_features=categorical_features,
    class_weights=catboost_weights, #applying the calculated weights
    early_stopping_rounds=30 
)

print("\nStarting CatBoost training with explicit class weights...")

cat_model_weighted.fit(X_train, y_train)
cat_y_pred_weighted = cat_model_weighted.predict(X_test)
cat_y_pred_weighted = cat_y_pred_weighted.flatten()

#calculate final metrics
cat_accuracy_weighted = accuracy_score(y_test, cat_y_pred_weighted)
cat_class_report_weighted = classification_report(y_test, cat_y_pred_weighted)

print(" Weighted CatBoost Model Training Complete!")
print(f"\n--- Weighted CatBoost Performance on Test Set ---")
print(f"Overall Accuracy: **{cat_accuracy_weighted:.4f}**")
print("\nClassification Report:\n", cat_class_report_weighted)

Calculated CatBoost Class Weights (Order: Cancer, Diabetic, Pneumonia): [4.11839530332681, 1.0, 2.3793103448275863]

Starting CatBoost training with explicit class weights...
 Weighted CatBoost Model Training Complete!

--- Weighted CatBoost Performance on Test Set ---
Overall Accuracy: **0.3613**

Classification Report:
               precision    recall  f1-score   support

      Cancer       0.16      0.25      0.19       438
    Diabetic       0.59      0.40      0.47      1804
   Pneumonia       0.24      0.34      0.28       758

    accuracy                           0.36      3000
   macro avg       0.33      0.33      0.32      3000
weighted avg       0.44      0.36      0.38      3000



In [12]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
from scipy.stats import loguniform, randint
from sklearn.svm import SVC
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.utils.class_weight import compute_class_weight

df = pd.read_csv('medical_conditions_dataset.csv')
df_cleaned = df.drop(columns=['id', 'full_name'])
numerical_cols = ['age', 'bmi', 'blood_pressure', 'glucose_levels']
df_cleaned['gender'] = df_cleaned['gender'].str.title()
df_cleaned['smoking_status'] = df_cleaned['smoking_status'].str.replace('-', ' ').str.title()
X_num = df_cleaned[numerical_cols]

# Imputation
imputer = KNNImputer(n_neighbors=5)
X_imputed_array = imputer.fit_transform(X_num)
X_imputed = pd.DataFrame(X_imputed_array, columns=numerical_cols)
X_cat = df_cleaned.drop(columns=numerical_cols).reset_index(drop=True)
X_final_cleaned = pd.concat([X_imputed, X_cat], axis=1)

#features and target seperation
X = X_final_cleaned.drop(columns=['condition'])
y_raw = X_final_cleaned['condition']

#encoding and scaling
X_encoded = pd.get_dummies(X, columns=['gender', 'smoking_status'], drop_first=True)
scaler = StandardScaler()
X_encoded[numerical_cols] = scaler.fit_transform(X_encoded[numerical_cols])
X_final = X_encoded

#target encoding
le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)
y = y_encoded

# Splitting data
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.3, random_state=42, stratify=y
)

#tuning setup
svc_model_tune = SVC(
    random_state=42,
    class_weight='balanced'
)

#SVC parameter space
param_dist_svc = {
    'C': loguniform(1e-1, 1e2), 
    'gamma': loguniform(1e-4, 1e-1), 
    'kernel': ['rbf', 'poly'], 
    'degree': randint(2, 4) 
}

#randomized search with high iteration count
random_search_svc = RandomizedSearchCV(
    estimator=svc_model_tune,
    param_distributions=param_dist_svc,
    n_iter=150,
    scoring='f1_weighted',
    cv=5,
    verbose=0,
    random_state=42,
    n_jobs=-1
)

print("Starting Intensive SVC Hyperparameter Tuning (Max Effort Scikit-learn)...")
random_search_svc.fit(X_train, y_train)

#extracting best model
best_svc_model = random_search_svc.best_estimator_
tuned_y_pred = best_svc_model.predict(X_test)

final_accuracy = accuracy_score(y_test, tuned_y_pred)
final_class_report = classification_report(y_test, tuned_y_pred)

print("intensive Tuning and final evaluation complete!")
print(f"\nBest parameters found: {random_search_svc.best_params_}")
print(f"Optimized SVC Accuracy on Test Set: **{final_accuracy:.4f}**")
print("\nClassification Report (Optimized Model):\n", final_class_report)

Starting Intensive SVC Hyperparameter Tuning (Max Effort Scikit-learn)...
intensive Tuning and Final Evaluation Complete!

Best parameters found: {'C': 0.19656332139612295, 'degree': 3, 'gamma': 0.026347470756444186, 'kernel': 'poly'}
Optimized SVC Accuracy on Test Set: **0.1460**

Classification Report (Optimized Model):
               precision    recall  f1-score   support

           0       0.15      1.00      0.25       438
           1       0.00      0.00      0.00      1804
           2       0.00      0.00      0.00       758

    accuracy                           0.15      3000
   macro avg       0.05      0.33      0.08      3000
weighted avg       0.02      0.15      0.04      3000



C:\Users\tejas\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\tejas\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\tejas\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from scipy.stats import uniform, randint
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

df = pd.read_csv('medical_conditions_dataset.csv')
df_cleaned = df.drop(columns=['id', 'full_name'])
numerical_cols = ['age', 'bmi', 'blood_pressure', 'glucose_levels']
df_cleaned['gender'] = df_cleaned['gender'].str.title()
df_cleaned['smoking_status'] = df_cleaned['smoking_status'].str.replace('-', ' ').str.title()
X_num = df_cleaned[numerical_cols]

imputer = KNNImputer(n_neighbors=5)
X_imputed_array = imputer.fit_transform(X_num)
X_imputed = pd.DataFrame(X_imputed_array, columns=numerical_cols)
X_cat = df_cleaned.drop(columns=numerical_cols).reset_index(drop=True)
X_final_cleaned = pd.concat([X_imputed, X_cat], axis=1)

X = X_final_cleaned.drop(columns=['condition'])
y = X_final_cleaned['condition']

# Define categorical features and weights 
categorical_features = ['gender', 'smoking_status']
class_weights_list = [4.11839530332681, 1.0, 2.3793103448275863] 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)


param_dist = {
    'depth': randint(4, 10),
    'learning_rate': uniform(0.01, 0.2), 
    'l2_leaf_reg': uniform(1, 10),
    'border_count': randint(32, 255),
    'bootstrap_type': ['Bernoulli'], 
    'subsample': uniform(0.6, 0.4), 
    'iterations': randint(500, 1500)
}

cat_model_tune = CatBoostClassifier(
    loss_function='MultiClass', 
    eval_metric='Accuracy', 
    random_state=42, 
    verbose=0,
    cat_features=categorical_features,
    class_weights=class_weights_list,
    task_type='CPU' 
)

random_search = RandomizedSearchCV(
    estimator=cat_model_tune,
    param_distributions=param_dist,
    n_iter=75,
    scoring='f1_weighted',
    cv=5,
    verbose=1,
    random_state=42,
)

print("Starting Intensive CatBoost Hyperparameter Tuning (CPU-Accelerated)...")
random_search.fit(X_train, y_train)

# extract best model and perform final evaluation
best_cat_model = random_search.best_estimator_
tuned_y_pred = best_cat_model.predict(X_test)
tuned_y_pred = tuned_y_pred.flatten()

# calculate final metrics
final_accuracy = accuracy_score(y_test, tuned_y_pred)
final_class_report = classification_report(y_test, tuned_y_pred)

print("Intensive Tuning and Final Evaluation Complete!")
print(f"\nBest parameters found: {random_search.best_params_}")
print(f"Optimized CatBoost Accuracy on Test Set: **{final_accuracy:.4f}**")
print("\nClassification Report (Optimized Model):\n", final_class_report)

Starting Intensive CatBoost Hyperparameter Tuning (CPU-Accelerated)...
Fitting 5 folds for each of 75 candidates, totalling 375 fits
